# Manufacturing LLM Fine-Tuning and Evaluation with LoRA / QLoRA

This Colab notebook fine-tunes **Qwen2.5-0.5B-Instruct** on 280 synthetic manufacturing cases, validates on 40, and evaluates on a separate 40-case test set.

It calculates structured accuracy, valid JSON rate, concept coverage, ROUGE, BERTScore, latency, GPU memory, validation loss, and perplexity. Groq is optional for LLM-as-a-Judge evaluation.

In [ ]:
%pip install -q transformers==4.46.3 datasets==3.1.0 accelerate==1.1.1 peft==0.13.2 trl==0.12.2 bitsandbytes==0.44.1 evaluate==0.4.3 rouge-score==0.1.2 bert-score==0.3.13 sentencepiece==0.2.0 groq==0.13.0 pandas==2.2.2 matplotlib==3.8.0 scikit-learn==1.5.2

In [ ]:
import gc, json, math, os, random, re, time
from pathlib import Path
import numpy as np, pandas as pd, torch
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from trl import SFTConfig, SFTTrainer
SEED=42; set_seed(SEED); random.seed(SEED)
BASE_MODEL="Qwen/Qwen2.5-0.5B-Instruct"; TRAINING_MODE="qlora"; OUTPUT_DIR="/content/manufacturing_adapter"
RESULTS_DIR=Path('/content/manufacturing_results'); RESULTS_DIR.mkdir(exist_ok=True)
def find_file(name):
 for p in [Path('data')/name,Path('/content')/name,Path('/content/data')/name]:
  if p.exists(): return p
train_file=find_file('manufacturing_train.jsonl'); validation_file=find_file('manufacturing_validation.jsonl'); test_file=find_file('manufacturing_test.jsonl')
if not all([train_file,validation_file,test_file]):
 from google.colab import files
 print('Upload the three split JSONL files'); files.upload()
 train_file=Path('/content/manufacturing_train.jsonl'); validation_file=Path('/content/manufacturing_validation.jsonl'); test_file=Path('/content/manufacturing_test.jsonl')

In [ ]:
def load_jsonl(path):
 return [json.loads(x) for x in Path(path).read_text(encoding='utf-8').splitlines() if x.strip()]
train_rows,validation_rows,test_rows=map(load_jsonl,[train_file,validation_file,test_file])
print(len(train_rows),len(validation_rows),len(test_rows)); print(json.dumps(train_rows[0],indent=2))

In [ ]:
SYSTEM_PROMPT='''You are a manufacturing maintenance and quality engineering assistant.
Return valid JSON only with exactly these keys:
{"severity":"low | medium | high | critical","component":"string","failure_category":"string","likely_causes":["string"],"validation_checks":["string"],"recommended_actions":["string"]}
Use only technically grounded information. Do not invent measurements.'''
tokenizer=AutoTokenizer.from_pretrained(BASE_MODEL,use_fast=True); tokenizer.pad_token=tokenizer.pad_token or tokenizer.eos_token
def row_to_text(r):
 msgs=[{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":f"{r['instruction']}\nManufacturing case: {r['case']}"},{"role":"assistant","content":json.dumps(r['reference_output'],ensure_ascii=False)}]
 return tokenizer.apply_chat_template(msgs,tokenize=False,add_generation_prompt=False)
train_ds=Dataset.from_list([{**r,'text':row_to_text(r)} for r in train_rows]); validation_ds=Dataset.from_list([{**r,'text':row_to_text(r)} for r in validation_rows])

In [ ]:
if not torch.cuda.is_available(): raise RuntimeError('Enable a Colab GPU runtime.')
dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
quant=None
if TRAINING_MODE=='qlora': quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=dtype)
model=AutoModelForCausalLM.from_pretrained(BASE_MODEL,quantization_config=quant,torch_dtype=None if quant else dtype,device_map='auto'); model.config.use_cache=False
peft_config=LoraConfig(r=16,lora_alpha=32,lora_dropout=.05,bias='none',task_type='CAUSAL_LM',target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
print(torch.cuda.get_device_name(0))

In [ ]:
def build_prompt(case):
 msgs=[{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":f"Diagnose the manufacturing issue and return the required JSON.\nManufacturing case: {case}"}]
 return tokenizer.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True)
@torch.inference_mode()
def generate_answer(m,case,max_new_tokens=240):
 inp=tokenizer(build_prompt(case),return_tensors='pt'); dev=next(m.parameters()).device; inp={k:v.to(dev) for k,v in inp.items()}
 out=m.generate(**inp,max_new_tokens=max_new_tokens,do_sample=False,repetition_penalty=1.05,pad_token_id=tokenizer.pad_token_id,eos_token_id=tokenizer.eos_token_id)
 return tokenizer.decode(out[0][inp['input_ids'].shape[1]:],skip_special_tokens=True).strip()
print(generate_answer(model,test_rows[0]['case']))

In [ ]:
torch.cuda.reset_peak_memory_stats(); start=time.perf_counter()
config=SFTConfig(output_dir=OUTPUT_DIR,dataset_text_field='text',max_seq_length=512,num_train_epochs=3,per_device_train_batch_size=2,per_device_eval_batch_size=2,gradient_accumulation_steps=4,learning_rate=2e-4,warmup_ratio=.1,lr_scheduler_type='cosine',logging_steps=5,eval_strategy='epoch',save_strategy='epoch',save_total_limit=2,gradient_checkpointing=True,fp16=not torch.cuda.is_bf16_supported(),bf16=torch.cuda.is_bf16_supported(),report_to='none',seed=SEED)
trainer=SFTTrainer(model=model,args=config,train_dataset=train_ds,eval_dataset=validation_ds,peft_config=peft_config,processing_class=tokenizer)
trainer.model.print_trainable_parameters(); trainer.train(); training_seconds=time.perf_counter()-start; peak_gpu_gb=torch.cuda.max_memory_allocated()/1024**3
trainer.model.save_pretrained(OUTPUT_DIR); tokenizer.save_pretrained(OUTPUT_DIR)
metrics=trainer.evaluate(); eval_loss=float(metrics['eval_loss']); perplexity=math.exp(eval_loss) if eval_loss<20 else float('inf')
print(training_seconds/60,peak_gpu_gb,eval_loss,perplexity)

In [ ]:
def predict(m,rows):
 ans=[]; lat=[]
 for i,r in enumerate(rows,1):
  s=time.perf_counter(); ans.append(generate_answer(m,r['case'])); lat.append(time.perf_counter()-s); print(f'{i}/{len(rows)}',end='\r')
 print(); return ans,lat
tuned_predictions,tuned_latencies=predict(trainer.model,test_rows)
pd.DataFrame({'case_id':[r['case_id'] for r in test_rows],'prediction':tuned_predictions}).to_csv(RESULTS_DIR/'tuned_predictions.csv',index=False)

In [ ]:
del trainer,model; gc.collect(); torch.cuda.empty_cache()
base_model=AutoModelForCausalLM.from_pretrained(BASE_MODEL,quantization_config=quant,torch_dtype=None if quant else dtype,device_map='auto'); base_model.eval()
base_predictions,base_latencies=predict(base_model,test_rows)
pd.DataFrame({'case_id':[r['case_id'] for r in test_rows],'prediction':base_predictions}).to_csv(RESULTS_DIR/'base_predictions.csv',index=False)

In [ ]:
def extract_json(text):
 try:return json.loads(text.strip())
 except:
  m=re.search(r'\{.*\}',text,re.S)
  if not m:return None
  try:return json.loads(m.group())
  except:return None
def norm(x):return str(x).strip().lower().replace(' ','_').replace('-','_')
def structured(preds,rows):
 parsed=[extract_json(p) for p in preds]; out={'valid_json_rate':np.mean([p is not None for p in parsed])}
 for field in ['severity','component','failure_category']:
  out[field+'_accuracy']=np.mean([p is not None and norm(p.get(field,''))==norm(r[field]) for p,r in zip(parsed,rows)])
 out['all_three_exact_accuracy']=np.mean([p is not None and all(norm(p.get(f,''))==norm(r[f]) for f in ['severity','component','failure_category']) for p,r in zip(parsed,rows)])
 return out
base_struct=structured(base_predictions,test_rows); tuned_struct=structured(tuned_predictions,test_rows)
def coverage(p,terms):return np.mean([t.lower() in p.lower() for t in terms])
base_cov=np.mean([coverage(p,r['required_terms']) for p,r in zip(base_predictions,test_rows)]); tuned_cov=np.mean([coverage(p,r['required_terms']) for p,r in zip(tuned_predictions,test_rows)])
print(base_struct,tuned_struct,base_cov,tuned_cov)

In [ ]:
import evaluate
from bert_score import score as bertscore
refs=[r['reference_text'] for r in test_rows]; rouge=evaluate.load('rouge')
base_rouge=rouge.compute(predictions=base_predictions,references=refs); tuned_rouge=rouge.compute(predictions=tuned_predictions,references=refs)
_,_,b=bertscore(base_predictions,refs,lang='en',verbose=False); _,_,t=bertscore(tuned_predictions,refs,lang='en',verbose=False)
base_bert=float(b.mean()); tuned_bert=float(t.mean())

In [ ]:
def lats(x):return {'mean':float(np.mean(x)),'p95':float(np.percentile(x,95))}
bl,tl=lats(base_latencies),lats(tuned_latencies)
summary=pd.DataFrame([{'model':'Base',**base_struct,'concept_coverage':base_cov,'rougeL':base_rouge['rougeL'],'bertscore_f1':base_bert,'mean_latency_sec':bl['mean'],'p95_latency_sec':bl['p95'],'validation_loss':np.nan,'perplexity':np.nan},{'model':'Fine-tuned',**tuned_struct,'concept_coverage':tuned_cov,'rougeL':tuned_rouge['rougeL'],'bertscore_f1':tuned_bert,'mean_latency_sec':tl['mean'],'p95_latency_sec':tl['p95'],'validation_loss':eval_loss,'perplexity':perplexity}])
summary.to_csv(RESULTS_DIR/'evaluation_summary.csv',index=False); display(summary.T)

In [ ]:
import matplotlib.pyplot as plt
cols=['valid_json_rate','severity_accuracy','component_accuracy','failure_category_accuracy','all_three_exact_accuracy','concept_coverage','rougeL','bertscore_f1']
ax=summary.set_index('model')[cols].T.plot(kind='bar',figsize=(14,6)); ax.set_ylim(0,1); ax.set_ylabel('Score'); ax.set_title('Base vs Fine-Tuned Manufacturing LLM'); plt.xticks(rotation=45,ha='right'); plt.tight_layout(); plt.savefig(RESULTS_DIR/'base_vs_tuned_metrics.png',dpi=160); plt.show()

## Optional Groq LLM-as-a-Judge

Add `GROQ_API_KEY` in Colab Secrets. This section is optional and incurs API usage. Report it as LLM-based evaluation, not expert validation.

In [ ]:
# Optional
# from google.colab import userdata
# from groq import Groq
# client = Groq(api_key=userdata.get("GROQ_API_KEY"))
# def judge(case, reference, answer, model_id="llama-3.1-8b-instant"):
#     prompt = (
#         "Evaluate this manufacturing answer. Return JSON only with 1-5 scores for "
#         "diagnostic_relevance, technical_completeness, safety_awareness, actionability, "
#         "factual_consistency, plus hallucination_risk 0-1 and rationale.\n"
#         f"CASE: {case}\nREFERENCE: {reference}\nANSWER: {answer}"
#     )
#     res = client.chat.completions.create(
#         model=model_id,
#         temperature=0,
#         response_format={"type": "json_object"},
#         messages=[{"role": "user", "content": prompt}],
#     )
#     return json.loads(res.choices[0].message.content)
# judgments = [judge(r["case"], r["reference_text"], pred) for r, pred in zip(test_rows, tuned_predictions)]
# pd.DataFrame(judgments).to_csv(RESULTS_DIR / "groq_judge_results.csv", index=False)


In [ ]:
from google.colab import files
import shutil
shutil.make_archive('/content/manufacturing_finetuning_outputs','zip','/content','manufacturing_results')
shutil.make_archive('/content/manufacturing_adapter','zip','/content','manufacturing_adapter')
print('/content/manufacturing_finetuning_outputs.zip'); print('/content/manufacturing_adapter.zip')
# files.download('/content/manufacturing_finetuning_outputs.zip')
# files.download('/content/manufacturing_adapter.zip')

## Interpretation

- Severity, component, and failure-category accuracy are labelled metrics.
- Valid JSON rate measures structured-output reliability.
- Concept coverage measures expected manufacturing terminology.
- ROUGE/BERTScore measure reference similarity, not engineering truth.
- The dataset is synthetic, so do not claim production performance.